In [ ]:
from __future__ import annotations
import sys; sys.path.insert(0, '..')

%load_ext autoreload
%autoreload 2


# python
import os
import ssl
import csv

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import session_info

from pathlib import Path
from inspect import cleandoc

# utils
from utils import Constants
from modules.preprocesing import preprocess
from modules import build_feature_pipeline

from wordcloud import WordCloud

# text
import re
import spacy
import unidecode
import pyLDAvis.gensim_models as gensimvis

# models
import umap
from sklearn.decomposition import PCA
from sklearn.decomposition import LatentDirichletAllocation

# sklearn
from sklearn.manifold import TSNE
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel, cosine_similarity, cosine_distances
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN

# Metrics
from sklearn.metrics import silhouette_score

# stat
from scipy import stats

# statsmodels
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# typings
from pandas import DataFrame as PandasDF
from typing import List, Dict, Union

# setup
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('max_colwidth', None)
# decimals
np.set_printoptions(precision=6)

In [ ]:
# cargar el dataset sklearn
if not os.environ.get('CI'):
    ssl._create_default_https_context =\
        ssl._create_unverified_context
          
# rutas absolutas
here: Path = Path.cwd().absolute().parent
data: Path = here / 'data'
poetry_fundation_cleaned: Path = data / 'CleanedPoetryFoundationData.csv'
cv_poetry: Path = data / 'vallejo_poems_en.csv'

In [ ]:
setup_load:Dict = dict(
    sep=Constants.PIPE_STR,
    quotechar=Constants.QUOTECHAR_STR,
    quoting=csv.QUOTE_NONNUMERIC,
    encoding=Constants.ENCODING
)

if not poetry_fundation_cleaned.is_file() or not cv_poetry.is_file():
    raise FileNotFoundError(
        cleandoc(f'''
        El archivo {poetry_fundation_cleaned} no existe.
        Por favor, descargue el archivo desde:
        https://www.kaggle.com/datasets/abhinavwalia95/poetryfoundationorg
        y coloquelo en la carpeta data.
        ''')
    )
    
poetry_df: PandasDF = (
    pd.read_csv(
        str(poetry_fundation_cleaned), 
        **setup_load
    ).dropna(subset=['title'])
)

cv_df: PandasDF = (
        pd.read_csv(
        str(cv_poetry), 
        **setup_load
    )
)

cv_df[['title', 'poem']] = (
        cv_df[['title', 'poem']]
        .apply(lambda col: col.astype(str).apply(preprocess))
    )

poetry_df = poetry_df.loc[~poetry_df.poem.isna(),:]

In [ ]:
display(poetry_df.head(3))

In [ ]:
display(cv_df.head(3))

In [ ]:
new_poems_df: PandasDF = (
    pd.concat(
        [
            poetry_df[['title', 'poem']],
            cv_df[['title', 'poem']]
        ],
        ignore_index=True
    )
    .dropna(subset=["title", "poem"])
    .reset_index(drop=True)
)

In [ ]:
feature_pipeline = build_feature_pipeline()
feature_pipeline

In [ ]:
X_features = feature_pipeline.fit_transform(
    new_poems_df["poem"].tolist()
)

X = X_features

# X representacion oficial no supervisada.
print("X_features shape:", X_features.shape)

In [ ]:
# 2. Vectorizar el texto
tfidf = TfidfVectorizer(max_features=5000) # Limitar a 5000 palabras más importantes
tfidf_matrix  = tfidf.fit_transform(new_poems_df.poem)

In [ ]:
df_tfidf = pd.DataFrame(
    tfidf_matrix.toarray(), 
    index=new_poems_df.title, 
    columns=tfidf.get_feature_names_out()
)
display(df_tfidf)

In [ ]:
cosine_sim = cosine_similarity(X_features)
cosine_sim

In [ ]:
def recomendador(title, cosine_sim=cosine_sim, df=new_poems_df):
    
    #Paso 2
    df = df.reset_index(drop=True)
    indices = pd.Series(df.index, index=df.title).drop_duplicates()
    #Paso 3
    idx = indices[title]

    #Paso 4
    sim_scores = list(enumerate(cosine_sim[idx]))

    #Paso 5
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    #Paso 6
    sim_scores = sim_scores[1:11]

    _indices = [i[0] for i in sim_scores]

    #Paso 7
    return pd.DataFrame(df.title.iloc[_indices])

In [ ]:
display(recomendador('black herald'))

In [ ]:
def recommendation_corr(
    query_title: str,
    corr_matrix: pd.DataFrame,
    top_n: int = 5
) -> pd.DataFrame:
    """
    Genera recomendaciones basadas en correlación de Pearson entre documentos.

    Args:
        query_title (str): Documento de referencia.
        corr_matrix (pd.DataFrame): Matriz de correlaciones.
        top_n (int): Número de recomendaciones a devolver.

    Returns:
        pd.DataFrame: Recomendaciones con puesto, título y similitud.
    """
    if query_title not in corr_matrix.columns:
        raise ValueError(f"El título '{query_title}' no está en la matriz de correlación.")

    # Extraer correlaciones con el documento de referencia
    sim_scores = corr_matrix[query_title].drop(query_title)

    # Ordenar de mayor a menor correlación
    sim_scores = sim_scores.sort_values(ascending=False).head(top_n)

    return pd.DataFrame({
        "Puesto": range(Constants.ONE, len(sim_scores) + Constants.ONE),
        "Recomendación": sim_scores.index,
        "Similitud": sim_scores.values
    })


df_features = pd.DataFrame(
    X_features,
    index=new_poems_df["title"]
)

corr_matrix = df_features.T.corr(method="pearson")

recom_features_corr = recommendation_corr(
    query_title='black herald',
    corr_matrix=corr_matrix,
    top_n=5
)

display(recom_features_corr)

In [ ]:
topic_vectorizer = CountVectorizer(
    stop_words="english",
    max_features=5000
)

X_topics = topic_vectorizer.fit_transform(
    new_poems_df["poem"].tolist()
)

# Modelo LDA con sklearn
lda_model = LatentDirichletAllocation(
    n_components=7,
    random_state=0
)

lda_topics = lda_model.fit_transform(X_topics)

# Identificar tema principal del poema 'black herald'
idx_cuento = new_poems_df[new_poems_df.title == 'black herald'].index[0]
tema_principal = lda_topics[idx_cuento].argmax()

# Probabilidades de pertenecer al tema principal
prob_tema_principal = lda_topics[:, tema_principal]

# Construir ranking de recomendaciones
df_recom = new_poems_df[["title"]].copy()
df_recom["prob_tema"] = prob_tema_principal

# eliminar el mismo poema 'black herald'
df_recom = df_recom[df_recom["title"] != 'black herald']

# ordenar por mayor probabilidad
df_recom = df_recom.sort_values("prob_tema", ascending=False)

# top N recomendaciones
top_recom = df_recom.head(5).reset_index(drop=True)
top_recom.insert(0, "Puesto", range(1, len(top_recom) + 1))

display(top_recom)

In [ ]:
# LDA  “top palabras por tópico”)
feature_names = topic_vectorizer.get_feature_names_out()

n_top = 10
top_words = []

for t, topic in enumerate(lda_model.components_):
    top_idx = topic.argsort()[::-1][:n_top]
    words = [feature_names[i] for i in top_idx]
    top_words.append((t, words))

display(
    pd.DataFrame({
        f"topic_{t}": words for t, words in top_words
    })
)

In [ ]:
# Modelo KMeans (+ silhouette)
def fit_kmeans_range(X, kmin=2, kmax=12, random_state=42):
    best = {"k": None, "sil": -1, "model": None, "labels": None}
    sils = []
    for k in range(kmin, kmax+1):
        km = KMeans(n_clusters=k, n_init="auto", random_state=random_state)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels, metric="cosine")
        sils.append((k, sil))
        if sil > best["sil"]:
            best = {"k": k, "sil": sil, "model": km, "labels": labels}
    return best, sils

X = X_features

best_km, km_sils = fit_kmeans_range(X)
new_poems_df["cluster_km"] = best_km["labels"]

best_km["k"], best_km["sil"]

In [ ]:
# Modelo Gaussian Mixture (GMM)
X_dense = X_features

n_components_pca = min(
    50,
    X_dense.shape[0] - 1,
    X_dense.shape[1]
)

Xpca = PCA(
    n_components=n_components_pca,
    random_state=42
).fit_transform(X_dense)

gmm = GaussianMixture(
    n_components=best_km["k"],
    covariance_type="full",
    random_state=42
)

gmm_labels = gmm.fit_predict(Xpca)
new_poems_df["cluster_gmm"] = gmm_labels

In [ ]:
# Moddel Agglomerative (en distancia coseno)

D = cosine_distances(X_features)         # distancia = 1 - coseno
agg = AgglomerativeClustering(
    n_clusters=best_km["k"],
    metric="precomputed",
    linkage="average"
)

agg_labels = agg.fit_predict(D)
new_poems_df["cluster_agg"] = agg_labels

In [ ]:
# Model DBSCAN (en espacio reducido con UMAP/t-SNE)
umap_ok = True
try:
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    X2 = reducer.fit_transform(X_features)  # acepta sparse
except Exception:
    umap_ok = False
    X2 = TSNE(n_components=2, random_state=42, metric="cosine").fit_transform(X_features)

db = DBSCAN(eps=0.5, min_samples=5, metric="euclidean").fit(X2)

new_poems_df["cluster_dbscan"] = db.labels_

In [ ]:
# Visualización 2D (matplotlib)
fig, ax = plt.subplots(figsize=(7,6))
sc = ax.scatter(X2[:,0], X2[:,1], c=new_poems_df["cluster_km"], s=18, alpha=0.2, cmap="Blues_r")
ax.set_title("Poemas en 2D ({} → clustering KMeans)".format("UMAP" if umap_ok else "t-SNE"))
ax.set_xlabel("comp-1"); ax.set_ylabel("comp-2")
plt.colorbar(sc, ax=ax, label="cluster_km")
plt.tight_layout(); plt.show()

### La apariencia “orgánica”:

Las ramificaciones son poemas que comparten similitudes con varios grupos → quedan como “puentes” o “brazos”.

Los nudos o concentraciones (zonas densas) son grupos de poemas con vocabulario/emoción muy parecida.

El hecho de que se vean como filamentos o bacterias es porque UMAP estira el espacio para mostrar continuidad entre regiones.

### Interpretación práctica

Si en el corpus hay poemas con temas/emociones muy conectados (por ejemplo, dolor ↔ muerte ↔ desesperanza en Vallejo), UMAP los hilvana en curvas continuas.

Si fueran más disjuntos (ej. poemas amorosos vs poemas políticos), verías islas separadas, no ramificaciones.

En poesía esto es natural: los temas no son rígidos, sino que fluyen de uno a otro. El gráfico refleja precisamente esa transición semántica difusa.

In [ ]:
# Similitud coseno (matriz y top-k vecinos)

# matriz de similitud (calculada)
# cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

# top-k por coseno (equivalente a recomendador)
nn = NearestNeighbors(metric="cosine", algorithm="brute")
nn.fit(X_features)
dists, idxs = nn.kneighbors(X_features, n_neighbors=6)  # incluye el propio (dist=0)
# ejemplo: vecinos del título 'black herald'
q = new_poems_df.index[new_poems_df.title=="black herald"][0]
vecinos = idxs[q][1:]  # sin el mismo
new_poems_df.title.iloc[vecinos]

## Integración de resultados: supervisado + no supervisado

Esta sección materializa el bloque **Integración de Resultados** del flujo del README. Combina:

- clusters no supervisados (`KMeans`, `GMM`, `Agglomerative`, `DBSCAN`),
- tópicos LDA,
- similitudes por coseno y correlación,
- predicciones supervisadas de etiquetas poéticas.

El objetivo es responder: **¿los clusters emergentes coinciden con emociones o temas predichos por el modelo supervisado?**

In [ ]:
# Imports adicionales para la integración supervisada/no supervisada
import ast
import warnings

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import jaccard_score, roc_auc_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)

### Preparar etiquetas supervisadas

El dataset de Poetry Foundation contiene etiquetas en la columna `tags`. Estas etiquetas se transforman a formato multilabel usando `MultiLabelBinarizer`, para entrenar un clasificador que luego predice etiquetas sobre el corpus integrado (`Poetry Foundation + Vallejo`).

In [ ]:
def parse_tags(value):
    """
    Convierte la columna tags a lista de etiquetas.
    Soporta valores que ya son listas o strings tipo "['Love', 'Nature']".
    """
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else [str(parsed)]
    except (ValueError, SyntaxError):
        return [
            tag.strip()
            for tag in str(value).split(Constants.COMMA_STR)
            if tag.strip()
        ]


poetry_df["tags"] = poetry_df["tags"].apply(parse_tags)

poetry_supervised_df: PandasDF = (
    poetry_df
    .loc[poetry_df["tags"].map(len) > 0, ["title", "poem", "tags"]]
    .dropna(subset=["poem"])
    .reset_index(drop=True)
)

display(poetry_supervised_df.head(3))

### Entrenar modelo supervisado multilabel

Se entrena una rama supervisada equivalente al flujo del README:

`build_feature_pipeline()` → `OneVsRestClassifier` → `StackingClassifier` → `ComplementNB + MultinomialNB` → `LogisticRegression`.

Este modelo se usa después para predecir etiquetas sobre todos los poemas integrados.

In [ ]:
seed: int = 42

mlb = MultiLabelBinarizer()
y_mlb = mlb.fit_transform(poetry_supervised_df["tags"])

X_train, X_test, y_train, y_test = train_test_split(
    poetry_supervised_df["poem"].tolist(),
    y_mlb,
    test_size=0.20,
    random_state=seed
)

pipeline_feature_sup: Pipeline = build_feature_pipeline()

pipeline_stack: Pipeline = Pipeline([
    ("Stack", OneVsRestClassifier(
        StackingClassifier(
            estimators=[
                ("cnb", ComplementNB(alpha=0.1)),
                ("mnb", MultinomialNB(alpha=0.099)),
            ],
            cv=KFold(n_splits=5, shuffle=True, random_state=seed),
            passthrough=True,
            final_estimator=LogisticRegression(
                random_state=seed,
                max_iter=1000
            )
        )
    ))
])

pipeline_full: Pipeline = Pipeline([
    ("features", pipeline_feature_sup),
    ("classifier", pipeline_stack),
])

pipeline_full.fit(X_train, y_train)

y_pred = pipeline_full.predict(X_test)

print(
    "Jaccard Score Micro:",
    round(jaccard_score(y_test, y_pred, average="micro"), 4)
)

try:
    y_score = pipeline_full.predict_proba(X_test)
    print(
        "ROC AUC Micro:",
        round(roc_auc_score(y_test, y_score, average="micro"), 4)
    )
except Exception as exc:
    print(f"No fue posible calcular ROC AUC con probabilidades: {exc}")

### Funciones auxiliares de similitud

Estas funciones generalizan las recomendaciones por similitud coseno y correlación de Pearson para todos los poemas. Luego sus salidas se integran en un único dataframe de resultados.

In [ ]:
def get_top_neighbors_by_cosine(
    cosine_sim: np.ndarray,
    df: pd.DataFrame,
    top_n: int = 5
) -> tuple[list[list[str]], list[list[float]]]:
    """
    Retorna los vecinos más similares por coseno para cada poema.
    """
    all_titles = df["title"].tolist()

    nearest_titles = []
    nearest_scores = []

    for idx in range(len(df)):
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        # Excluir el mismo poema
        sim_scores = sim_scores[1: top_n + 1]

        titles = [all_titles[i] for i, _ in sim_scores]
        scores = [round(score, 4) for _, score in sim_scores]

        nearest_titles.append(titles)
        nearest_scores.append(scores)

    return nearest_titles, nearest_scores


def get_top_neighbors_by_corr(
    corr_matrix: pd.DataFrame,
    top_n: int = 5
) -> tuple[list[list[str]], list[list[float]]]:
    """
    Retorna vecinos más similares por correlación de Pearson para cada poema.
    """
    nearest_titles = []
    nearest_scores = []

    for title in corr_matrix.columns:
        sim_scores = corr_matrix[title].drop(title)
        sim_scores = sim_scores.sort_values(ascending=False).head(top_n)

        nearest_titles.append(sim_scores.index.tolist())
        nearest_scores.append(sim_scores.round(4).tolist())

    return nearest_titles, nearest_scores


nearest_titles_cosine, nearest_scores_cosine = get_top_neighbors_by_cosine(
    cosine_sim=cosine_sim,
    df=new_poems_df,
    top_n=5
)

nearest_titles_corr, nearest_scores_corr = get_top_neighbors_by_corr(
    corr_matrix=corr_matrix,
    top_n=5
)

### Construir `df_integracion`

Este dataframe integra los resultados principales del análisis:

- clusters (`cluster_km`, `cluster_gmm`, `cluster_agg`, `cluster_dbscan`),
- tópico dominante LDA,
- términos representativos del tópico,
- etiquetas predichas por el modelo supervisado,
- vecinos por similitud coseno y correlación,
- coordenadas 2D para visualización.

In [ ]:
# Integración de Resultados

# 1. Predicciones supervisadas sobre todo el corpus integrado
pred_tags = mlb.inverse_transform(
    pipeline_full.predict(new_poems_df["poem"].tolist())
)

# 2. Topico dominante por poema
lda_topic_id = lda_topics.argmax(axis=1)
lda_topic_prob = lda_topics.max(axis=1)

topic_terms = {
    topic_id: Constants.COMMA_STR.join(words[:5])
    for topic_id, words in top_words
}

# 3. DataFrame integrado
df_integracion: PandasDF = pd.DataFrame({
    "title": new_poems_df["title"].values,

    # Clusters no supervisados
    "cluster_km": new_poems_df["cluster_km"].values,
    "cluster_gmm": new_poems_df["cluster_gmm"].values,
    "cluster_agg": new_poems_df["cluster_agg"].values,
    "cluster_dbscan": new_poems_df["cluster_dbscan"].values,

    # LDA
    "lda_topic_id": lda_topic_id,
    "lda_topic_prob": lda_topic_prob.round(4),
    "lda_topic_terms": [
        topic_terms.get(topic_id, "")
        for topic_id in lda_topic_id
    ],

    # Supervisado
    "predicted_tags": list(map(list, pred_tags)),

    # Similitud coseno
    "nearest_titles_cosine": nearest_titles_cosine,
    "nearest_scores_cosine": nearest_scores_cosine,

    # Correlacion Pearson
    "nearest_titles_corr": nearest_titles_corr,
    "nearest_scores_corr": nearest_scores_corr,

    # Coordenadas 2D
    "embedding_x": X2[:, 0],
    "embedding_y": X2[:, 1],
})

display(df_integracion.head(10))

### Clusters vs. etiquetas supervisadas

Este cruce permite contrastar si los clusters emergentes del análisis no supervisado coinciden con las etiquetas predichas por el modelo supervisado.

In [ ]:
df_exploded = (
    df_integracion
    .explode("predicted_tags")
    .dropna(subset=["predicted_tags"])
)

cluster_tag_matrix = pd.crosstab(
    df_exploded["cluster_km"],
    df_exploded["predicted_tags"],
    normalize="index"
).round(3)

display(cluster_tag_matrix)

In [ ]:
cluster_tag_counts = pd.crosstab(
    df_exploded["cluster_km"],
    df_exploded["predicted_tags"]
)

display(cluster_tag_counts)

### Clusters vs. tópicos LDA

Este cruce permite observar si los clusters geométricos se alinean con tópicos latentes del corpus.

In [ ]:
cluster_topic_matrix = pd.crosstab(
    df_integracion["cluster_km"],
    df_integracion["lda_topic_id"],
    normalize="index"
).round(3)

display(cluster_topic_matrix)

cluster_topic_summary = (
    df_integracion
    .groupby(["cluster_km", "lda_topic_id", "lda_topic_terms"])
    .size()
    .reset_index(name="count")
    .sort_values(["cluster_km", "count"], ascending=[True, False])
)

display(cluster_topic_summary)

### Resumen interpretativo por cluster

Esta tabla muestra las etiquetas supervisadas más frecuentes dentro de cada cluster no supervisado.

In [ ]:
cluster_tag_summary = (
    df_exploded
    .groupby(["cluster_km", "predicted_tags"])
    .size()
    .reset_index(name="count")
    .sort_values(["cluster_km", "count"], ascending=[True, False])
)

top_tags_by_cluster = (
    cluster_tag_summary
    .groupby("cluster_km")
    .head(5)
    .reset_index(drop=True)
)

display(top_tags_by_cluster)

### Visualización integrada

La visualización mantiene la proyección 2D de UMAP/t-SNE, coloreada por `cluster_km`, y resalta poemas asociados a Vallejo o a *black herald* cuando están presentes en el corpus.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

sc = ax.scatter(
    df_integracion["embedding_x"],
    df_integracion["embedding_y"],
    c=df_integracion["cluster_km"],
    s=18,
    alpha=0.25,
    cmap="Blues_r"
)

ax.set_title(
    "Integración no supervisada: UMAP/t-SNE + KMeans + etiquetas supervisadas"
)
ax.set_xlabel("comp-1")
ax.set_ylabel("comp-2")

plt.colorbar(sc, ax=ax, label="cluster_km")

# Anotar poemas de Vallejo si están en el dataframe
mask_vallejo = df_integracion["title"].str.contains(
    "black herald|vallejo",
    case=False,
    na=False
)

for _, row in df_integracion[mask_vallejo].iterrows():
    ax.annotate(
        row["title"],
        xy=(row["embedding_x"], row["embedding_y"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=9,
        weight="bold"
    )

plt.tight_layout()
plt.show()

### Exportar resultados integrados

Se guarda un CSV con la integración de clusters, tópicos, similitudes y etiquetas predichas. Este archivo puede usarse luego para análisis externo, visualización o documentación en el README.

In [ ]:
output_path = here / "data" / "integrated_results.csv"

df_integracion.to_csv(
    output_path,
    index=False,
    encoding=Constants.ENCODING,
    sep=Constants.PIPE_STR
)

print(f"Archivo generado: {output_path}")

### Visualización completa del pipeline

GitHub no renderiza diagramas HTML interactivos generados por scikit-learn.
Para ver la representación visual completa del pipeline, utiliza **nbviewer**:

[04_embeddings_unsupervised.ipynb](https://nbviewer.org/github/HubertRonald/VersoVector/blob/main/notebook/04_embeddings_unsupervised.ipynb)